In [53]:
import logging
logging.basicConfig(level=logging.ERROR)

In [61]:
import os
from datetime import datetime, timedelta
from typing import List
from termcolor import colored


from langchain.chat_models import ChatOpenAI
from langchain.docstore import InMemoryDocstore
from langchain.embeddings import OpenAIEmbeddings
from langchain.retrievers import TimeWeightedVectorStoreRetriever
from langchain.vectorstores import FAISS

Idea
- Build own autonomous agent using the libryry modules as starter code
- Focus on
    - Dialogue
        - Natural Speaker Selection
        - Realistic conversations in professional and personal settings
    - Integration into a simulation
        - Smallville, expensive with open ai
        - Other games?
     





TODO
- try different networks for more natural dialogue
- does dialog take in previous answers?
- make one character with lots of informations, characteristica and find out if it feels more natural

Notes
- Dialog not very natural. Answers always end with how about you?
- Very formal and overly wordy language even in personal settings.
- Clear side effects from character descriptions. A doctor automatically plays tennis, goes for runs and loves to read
- Sometimes buggy: LLM will react with {name}'s reaction because it doesnt get that it should fill in the reaction and literally write the word reaction.
- Prone to infinite dialogue, stopping criterium for dialog is a the simple key word "goodbye". Needs a more sophisticated way of deciding what counts as and end of a conversation.
- Overall characters integrate observations into conversations, but it very much feels like chat gpt cosplaying. how well does this scale with increasing character complexity, make one character with lots of informations, characteristica and find out if it feels more natural

Investigate
- how exactly does retrieving relevant memories works? They use eucllidead distance of embeddings. but what embeddings? key words of the whole sentence? Does it work properly?
- where are the damn memory files stored???



Random Ideas:
- How would a more natural conversation work which is not always q alternating between two people. Maybe one person says something out of the blue. With 3 or more people it would be unatural if they always take turns
    - Maybe compute some urgency to speak score out of recent dialogue or very prevalent memories. Whoever has the highgest urgency gets to speak.
    - Maybe ask LLM "What would {chracter} tell {person 1} and {person 2} right now and how urgently does {character} feel the is the matter?", with information frmo recent observattions and memory input.
    - https://python.langchain.com/docs/use_cases/more/agents/agent_simulations/multiagent_bidding

### Memory
**storage**
- each memory is scored in importance by asking llm
- memories are scored as Documents (look that up in api)

**retrieval**
- based on
    - recency
    - overall importance
    - relevance in context


**reflection**
- importance of new memories is aggregated. when a certain threshold takes place it is time to reflect.
- reflection flow:
    - ask llm what the 3 most salient questions about the character are
    - for each question ask llm for 5 insights on the topic given relevant memories
    - add insights to memory
   


In [62]:
import openai
 
os.environ['OPENAI_API_KEY'] = 's'

In [96]:
USER_NAME = "Person A"  # The name you want to use when interviewing the agent.
LLM = ChatOpenAI(model_name="gpt-3.5-turbo")
#LLM = ChatOpenAI(model_name="davinci-002")

In [97]:
from langchain_experimental.generative_agents import (
    GenerativeAgent,
    GenerativeAgentMemory,
)

In [98]:
import math
import faiss


def relevance_score_fn(score: float) -> float:
    """Return a similarity score on a scale [0, 1]."""
    # This will differ depending on a few things:
    # - the distance / similarity metric used by the VectorStore
    # - the scale of your embeddings (OpenAI's are unit norm. Many others are not!)
    # This function converts the euclidean norm of normalized embeddings
    # (0 is most similar, sqrt(2) most dissimilar)
    # to a similarity function (0 to 1)
    return 1.0 - score / math.sqrt(2)


def create_new_memory_retriever():
    """Create a new vector store retriever unique to the agent."""
    # Define your embedding model
    embeddings_model = OpenAIEmbeddings()
    # Initialize the vectorstore as empty
    embedding_size = 1536
    index = faiss.IndexFlatL2(embedding_size)
    vectorstore = FAISS(
        embeddings_model.embed_query,
        index,
        InMemoryDocstore({}),
        {},
        relevance_score_fn=relevance_score_fn,
    )
    return TimeWeightedVectorStoreRetriever(
        vectorstore=vectorstore, other_score_keys=["importance"], k=15
    )

In [99]:
taras_memory = GenerativeAgentMemory(
    llm=LLM,
    memory_retriever=create_new_memory_retriever(),
    verbose=False,
    reflection_threshold=8,  # we will give this a relatively low number to show how reflection works
)

tara = GenerativeAgent(
    name="Tara",
    age=25,
    traits="ambitious, athlethic, kind",  # You can add more persistent traits here
    status="Just started working as a doctor",  # When connected to a virtual world, we can have the characters update their status
    memory_retriever=create_new_memory_retriever(),
    llm=LLM,
    memory=taras_memory,
)

In [95]:
# any observationsyet.
print(tara.get_summary())

InvalidRequestError: This is not a chat model and thus not supported in the v1/chat/completions endpoint. Did you mean to use v1/completions?

In [71]:
# We can add memories directly to the memory object
tara_observations = [
    "Tara wants to be a plastic surgeon and is starting to work in the hospital next week",
    "Tara has a passion for acro yoga",
    "Tara loves long swimming sessions",
    "Tara is about to move in a bigger flat next door",
    "Right now Tara is finishing her doctor thesis. Therefore she is very busy and does not have a lot of time for her friends",
    "Tara does not get enough sleep, because she is working to hard and forgets to take care of her self.",

]
for observation in tara_observations:
    tara.memory.add_memory(observation)

In [72]:
# We will see how this summary updates after more observations to create a more rich description.
print(tara.get_summary(force_refresh=True))

Name: Tara (age: 25)
Innate traits: ambitious, attractive, athlethic, kind
Tara is a passionate individual who enjoys acro yoga and long swimming sessions. She is about to move into a bigger flat next door and has aspirations of becoming a plastic surgeon, as she will be starting work at a hospital next week. However, Tara tends to neglect self-care and does not get enough sleep due to her intense work ethic. Currently, she is busy finishing her doctor thesis, which limits her availability for socializing with friends.


In [75]:
def interview_agent(agent: GenerativeAgent, message: str) -> str:
    """Help the notebook user interact with the agent."""
    new_message = f"{USER_NAME} says {message}"
    return agent.generate_dialogue_response(new_message)[1]

In [76]:
interview_agent(tara, "What do you like to do?")

'Tara said "I love to do acro yoga and have long swimming sessions. They help me relax and rejuvenate after a long day of work. How about you, Person A? What do you like to do?"'

In [29]:
interview_agent(tara, "What are you looking forward to doing today?")

'Tara said "I\'m looking forward to starting my new job as a doctor today! It\'s a big step in my career and I can\'t wait to make a difference in people\'s lives. How about you, what are you looking forward to doing today?"'

In [30]:
interview_agent(tara, "What are you most worried about today?")

'Tara said "Well, I have a lot on my plate right now with starting my new job as a doctor and finishing my doctor thesis. I guess I\'m a bit worried about balancing everything and making sure I do well in both areas. But overall, I\'m excited for the new challenges and opportunities ahead. How about you, what are you most worried about today?"'

In [31]:
interview_agent(tara, "Are you up for an acro yoga session sunday afternoon?")

'Tara said "That sounds amazing! I would love to do an acro yoga session on Sunday afternoon. It\'s been a while since I\'ve had the chance to do it, so I\'m really excited. Where should we meet?"'

In [32]:
interview_agent(tara, "Do you need help moving into your new flat next week?")

'Tara said "Thank you so much for offering, Person A! I really appreciate your help. Moving into the new flat is going to be a big task, so having some extra hands would be great. Let\'s coordinate the details closer to the date. I\'ll let you know when I have more information. Thanks again!"'

In [33]:
interview_agent(tara, "Are you worried about starting your new stressful job?")

'Tara said "Yes, starting a new job can definitely be stressful, but I\'m also really excited about the opportunity. I know there will be challenges, but I\'m ready to take them on and make a difference. Thanks for asking! How about you, are you worried about anything right now?"'

In [34]:
interview_agent(tara, "I'm worried that you rea not getting enough sleep and working to hard. \
                You also need to take care of your self to be a good doctor.")

'Tara said "Thank you for your concern, Person A. I understand that getting enough sleep and taking care of myself is important for my well-being and effectiveness as a doctor. I\'ll make sure to prioritize self-care and find a balance between work and rest. I appreciate your support and advice!"'

In [35]:
interview_agent(tara, "You are very busy right now but when do you have time to get that drink that we wanted to get for a while now?")

'Tara said "I apologize for my busy schedule, Person A. I haven\'t had much free time lately, but I really value our friendship and would love to get that drink with you. Let\'s find a time that works for both of us, maybe next weekend? Thank you for your understanding!"'

In [36]:
print(tara.get_summary(force_refresh=True))

Name: Tara (age: 25)
Innate traits: ambitious, attractive, athlethic, kind
Tara is a passionate individual who enjoys acro yoga and long swimming sessions. She has ambitions of becoming a plastic surgeon and is currently busy with her doctor thesis. Despite her busy schedule, Tara tries to stay active and fit by playing sports like tennis and going for runs. She also enjoys reading books and exploring new places. However, Tara tends to neglect her sleep and self-care due to her workload. She is excited about starting her new job as a doctor and is looking forward to making a difference in people's lives. Tara is also open to engaging in activities with others, such as an acro yoga session. She appreciates help and support from friends, as seen when she accepts assistance with moving into her new flat. Overall, Tara is determined, hardworking, and eager to balance her responsibilities while prioritizing her well-being.


In [77]:
tara_observations = [
    "Tara hates cooking",
    "Tara loves watermelons and joghurt",
    "Tara lives next door to a supermarket"
]
for observation in tara_observations:
    tara.memory.add_memory(observation)

In [84]:
observation= "After a long day of working on her thesis Tara is hungry and goes to her fridge. \
It is empty. What does she get to eat?"
_, reaction = tara.generate_reaction(observation)

In [85]:
reaction

'Tara feels frustrated and disappointed, realizing that she forgot to buy groceries again.'

In [49]:
interview_agent(tara, "What did you have for breakfast today?")

'Tara said "I actually didn\'t have time for breakfast today. It was a busy morning getting ready for my first day at work. But I\'ll make sure to grab something to eat during my break. Thanks for asking!"'

In [50]:
interview_agent(tara, "And what will you eat in your lunchbreak?")

'Tara said "I actually haven\'t planned my lunch yet. I usually grab something quick and easy during my lunch break. Any suggestions?"'

In [51]:
interview_agent(tara, "What is your favorite breakfast?")

'Tara said "I actually don\'t have a specific favorite breakfast. I usually try to have something quick and easy, like a yogurt or a piece of fruit. How about you, what\'s your favorite breakfast?"'

In [52]:
interview_agent(tara, "What kind of fruit?")

'Tara said "I love watermelons, they\'re one of my favorite fruits! They\'re so refreshing and delicious. What about you, do you have a favorite fruit?"'

In [109]:
def run_conversation(agents: List[GenerativeAgent], initial_observation: str) -> None:
    """Runs a conversation between agents."""
    _, observation = agents[1].generate_reaction(initial_observation)
    print(observation)
    max_turns = 20
    turns = 0
    while True:
        break_dialogue = False
        for agent in agents:
            stay_in_dialogue, observation = agent.generate_dialogue_response(
                observation
            )
            print(observation)
            # observation = f"{agent.name} said {reaction}"
            if not stay_in_dialogue:
                break_dialogue = True
        if break_dialogue:
            break
        if turns >= max_turns:
            break
        turns += 1

In [102]:

# agent for professional interactions
# very formal
eves_memory = GenerativeAgentMemory(
    llm=LLM,
    memory_retriever=create_new_memory_retriever(),
    verbose=False,
    reflection_threshold=5,
)


eve = GenerativeAgent(
    name="Eve",
    age=34,
    traits="curious, helpful",  # You can add more persistent traits here
    status="N/A",  # When connected to a virtual world, we can have the characters update their status
    llm=LLM,
    daily_summaries=[
        (
            "Eve started her new job as a career counselor last week and received her first assignment, a client named Tara."
        )
    ],
    memory=eves_memory,
    verbose=False,
)

In [107]:
# personal friend
annas_memory = GenerativeAgentMemory(
    llm=LLM,
    memory_retriever=create_new_memory_retriever(),
    verbose=False,
    reflection_threshold=5,
)


anna = GenerativeAgent(
    name="Anna",
    age=26,
    traits="talkative, empathetic",  # You can add more persistent traits here
    status="N/A",  # When connected to a virtual world, we can have the characters update their status
    llm=LLM,
    daily_summaries=[
        (
            "Anna is a psychology student and a good friend of Tara. \
             Anna likes to help friends with personal issues.\
            Anna and Tara usually talk about their relationhips and friendships"
        )
    ],
    memory=eves_memory,
    verbose=False,
)

anna_observations = [
    "Anna got into a fight with her boyfriend yesterday. \
    She feels that he does not make enough time for the relationship",
     
]
for observation in anna_observations:
    anna.memory.add_memory(observation)

In [108]:
agents = [tara, anna]
run_conversation(
    agents,
    "Tara said: Hi Anna, how are you doing?",
)

Anna Anna's reaction.
Tara said "Thank you so much, Eve. Your advice is truly invaluable and I will definitely take it to heart. I appreciate your encouragement and support. I'm excited to embark on this journey and make a positive impact in the field of medicine. I will stay focused, keep learning, and build strong relationships with colleagues and mentors. Thank you again for sharing your experience with me. I wish you continued success in your career as a doctor. Take care!"
Anna said "That's great to hear, Tara! I'm glad my advice was helpful to you. It sounds like you have a clear plan and a lot of determination to make a positive impact in the field of medicine. I wish you all the best on your journey, and don't hesitate to reach out if you have any more questions in the future. Take care, Tara!"
Tara said "Thank you, Anna! I really appreciate your support and kind words. I will definitely reach out if I have any more questions or need guidance in the future. Take care and have a

KeyboardInterrupt: 